# Create Israel Science Foundation Awards

Creates OpenAlex award rows from the official Israel Science Foundation (ISF) public site API.

**Prerequisites:**
- Admin/human with AWS credentials runs `scripts/local/isf_to_s3.py` to download the official API response, write parquet, and upload it.
- Contractor has prepared the script, notebook, registry entry, tracker row, and local QA, but does not have S3/Databricks access.
- The downloader writes parquet columns as strings per `plans/awards/how-to-add-a-funder.md`; this notebook does type conversion with `TRY_CAST` / `TRY_TO_DATE`.

**Data source:** `https://www.isf.org.il/api/grant/GetGrantSearchData` as called by the official public ISF Angular application for Research Grants, Scientific Equipment, Workshops, Publications, and Postdoctoral Fellowships pages.  
**S3 location:** `s3a://openalex-ingest/awards/isf/isf_grants.parquet`  
**Local full-source validation on 2026-05-27:** 17,017 official API rows from 175 allocation/year slices (1991-2025 requests; real rows 2000-2025), deduped to 16,194 unique `applicationId` awards by keeping the row with the source-published positive `approvedSum` over zero-amount duplicates. 97.1% amount/currency coverage from `approvedSum` (source UI labels this as Annual Budget); 100% title/display name, institution, lead PI, year, program, and landing-page coverage. Total ILS 3,683,233,858 across rows with positive annual budgets.

**OpenAlex funder mapping:**
- funder_id: 4320322252
- display_name: `Israel Science Foundation`
- ROR: `https://ror.org/04sazxf24`
- DOI: `10.13039/501100003977`

**Mapping summary:** One OpenAlex award row per deduped ISF `applicationId`. `amount` is the official source-published approved annual budget when positive, with `currency='ILS'`; source rows with `approvedSum=0` are left as NULL amount/currency. `funder_scheme` is the ISF program/grant type. `funding_type` comes from the source allocation channel: research grants as `research`, postdoctoral fellowships as `fellowship`, and equipment/workshop/publication support as `grant`.


## Step 1: Create Raw Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.isf_raw
USING delta
AS
SELECT
    *,
    current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/isf/isf_grants.parquet`;


In [ ]:
%sql
-- Check raw row count. Local validation on 2026-05-27 found 16,194 rows.
SELECT COUNT(*) as total_isf_grants
FROM openalex.awards.isf_raw;


In [ ]:
%sql
-- Verify actual column names before transform logic references them.
DESCRIBE openalex.awards.isf_raw;


In [ ]:
%sql
-- Sample raw ISF data.
SELECT
    funder_award_id,
    application_id,
    display_name,
    grant_type_name,
    source_allocation_type_name,
    source_year,
    amount,
    currency,
    lead_investigator_name,
    institution,
    landing_page_url
FROM openalex.awards.isf_raw
LIMIT 10;


## Step 1.5: Inspect Raw Data, Money, Dates, and Native Keys

In [ ]:
%sql
-- Money-field scan per runbook Step 1.5. Uses information_schema, not DESCRIBE as a subquery.
SELECT column_name
FROM openalex.information_schema.columns
WHERE table_schema = 'awards'
  AND table_name = 'isf_raw'
  AND LOWER(column_name) RLIKE
    'amount|amt|total|value|sum|funded|funding|cost|budget|grant|award|approved|currency|ils';


In [ ]:
%sql
-- Confirm amount/date coverage before mapping.
SELECT
    COUNT(*) AS total,
    COUNT(amount) AS rows_with_amount,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) AS pct_with_amount,
    ROUND(MIN(TRY_CAST(amount AS DOUBLE)), 0) AS min_amount,
    ROUND(MAX(TRY_CAST(amount AS DOUBLE)), 0) AS max_amount,
    ROUND(SUM(TRY_CAST(amount AS DOUBLE)), 0) AS total_amount_ils,
    COUNT(start_date) AS rows_with_start_date,
    COUNT(end_date) AS rows_with_end_date,
    MIN(TRY_CAST(source_year AS INT)) AS min_year,
    MAX(TRY_CAST(source_year AS INT)) AS max_year
FROM openalex.awards.isf_raw;


In [ ]:
%sql
-- Native-key inspection.
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT funder_award_id) AS distinct_funder_award_ids,
    COUNT(DISTINCT application_id) AS distinct_application_ids,
    COUNT(DISTINCT source_record_hash) AS distinct_source_rows
FROM openalex.awards.isf_raw;


In [ ]:
%sql
-- Allocation/year distribution.
SELECT
    source_year,
    source_allocation_type_name,
    COUNT(*) AS cnt,
    ROUND(SUM(TRY_CAST(amount AS DOUBLE)), 0) AS total_ils
FROM openalex.awards.isf_raw
GROUP BY source_year, source_allocation_type_name
ORDER BY TRY_CAST(source_year AS INT) DESC, cnt DESC;


In [ ]:
%sql
-- Program and person/institution coverage.
SELECT
    COUNT(*) AS total,
    COUNT(grant_type_name) AS has_program,
    COUNT(lead_investigator_name) AS has_pi_raw,
    COUNT(lead_investigator_given_name) AS has_pi_given,
    COUNT(lead_investigator_family_name) AS has_pi_family,
    COUNT(institution) AS has_institution,
    COUNT(description) AS has_keywords_description,
    COUNT(additional_investigators_raw) AS has_additional_investigators
FROM openalex.awards.isf_raw;


## Step 1.6: Funder Existence Fail-Fast

In [ ]:
%sql
-- Must return exactly 1 row. If this is empty, STOP: the funder is missing from openalex.common.funder.
SELECT funder_id, display_name, ror_id, doi, country_code
FROM openalex.common.funder
WHERE funder_id = 4320322252;


## Step 2: Transform to OpenAlex Award Schema

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.isf_awards
USING delta
AS
WITH
isf_funder AS (
    SELECT
        funder_id,
        display_name,
        ror_id,
        doi
    FROM openalex.common.funder
    WHERE funder_id = 4320322252
),

raw_prepared AS (
    SELECT
        *,
        LOWER(TRIM(funder_award_id)) AS native_award_id,
        TRY_CAST(amount AS DOUBLE) AS parsed_amount,
        CASE WHEN TRY_CAST(amount AS DOUBLE) IS NOT NULL THEN 'ILS' ELSE NULL END AS parsed_currency,
        TRY_TO_DATE(start_date, 'yyyy-MM-dd') AS parsed_start_date,
        TRY_TO_DATE(end_date, 'yyyy-MM-dd') AS parsed_end_date,
        TRY_CAST(source_year AS INT) AS parsed_start_year,
        TRY_CAST(years AS INT) AS parsed_duration_years
    FROM openalex.awards.isf_raw
    WHERE funder_award_id IS NOT NULL
      AND TRIM(funder_award_id) != ''
      AND display_name IS NOT NULL
      AND TRIM(display_name) != ''
),

awards_transformed AS (
    SELECT
        abs(xxhash64(CONCAT(f.funder_id, ':', g.native_award_id))) % 9000000000 as id,

        TRIM(g.display_name) as display_name,

        CASE
            WHEN g.description IS NULL OR TRIM(g.description) = '' THEN NULL
            ELSE TRIM(g.description)
        END as description,

        f.funder_id,
        g.native_award_id as funder_award_id,

        g.parsed_amount as amount,
        g.parsed_currency as currency,

        struct(
            CONCAT('https://openalex.org/F', f.funder_id) as id,
            f.display_name,
            f.ror_id,
            f.doi
        ) as funder,

        COALESCE(NULLIF(TRIM(g.funding_type), ''), 'grant') as funding_type,

        COALESCE(
            NULLIF(TRIM(g.grant_type_name), ''),
            NULLIF(TRIM(g.source_allocation_type_name), ''),
            'ISF grant'
        ) as funder_scheme,

        'isf_grant_search' as provenance,

        g.parsed_start_date as start_date,
        g.parsed_end_date as end_date,
        COALESCE(YEAR(g.parsed_start_date), g.parsed_start_year) as start_year,
        COALESCE(YEAR(g.parsed_end_date), g.parsed_start_year + g.parsed_duration_years - 1) as end_year,

        struct(
            NULLIF(TRIM(g.lead_investigator_given_name), '') as given_name,
            NULLIF(TRIM(g.lead_investigator_family_name), '') as family_name,
            CAST(NULL AS STRING) as orcid,
            g.parsed_start_date as role_start,
            struct(
                NULLIF(TRIM(g.institution), '') as name,
                'IL' as country,
                CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) as ids
            ) as affiliation
        ) as lead_investigator,

        CAST(NULL AS STRUCT<
            given_name:STRING,
            family_name:STRING,
            orcid:STRING,
            role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >) as co_lead_investigator,

        CAST(NULL AS ARRAY<STRUCT<
            given_name:STRING,
            family_name:STRING,
            orcid:STRING,
            role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >>) as investigators,

        NULLIF(TRIM(g.landing_page_url), '') as landing_page_url,
        CAST(NULL AS STRING) as doi,

        CONCAT('https://api.openalex.org/works?filter=awards.id:G',
               CAST(abs(xxhash64(CONCAT(f.funder_id, ':', g.native_award_id))) % 9000000000 AS STRING)) as works_api_url,

        current_timestamp() as created_date,
        current_timestamp() as updated_date

    FROM raw_prepared g
    CROSS JOIN isf_funder f
)
SELECT *
FROM awards_transformed;


## Step 3: Delete Previous Source Rows and Insert Priority 146

In [ ]:
%sql
-- Remove previous ISF data before inserting fresh data.
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'isf_grant_search' AND priority = 146;

-- Insert into openalex_awards_raw with exact OpenAlex awards schema plus priority.
INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id,
    display_name,
    description,
    funder_id,
    funder_award_id,
    amount,
    currency,
    funder,
    funding_type,
    funder_scheme,
    provenance,
    start_date,
    end_date,
    start_year,
    end_year,
    lead_investigator,
    co_lead_investigator,
    investigators,
    landing_page_url,
    doi,
    works_api_url,
    created_date,
    updated_date,
    146 as priority
FROM openalex.awards.isf_awards;


## Step 6: Verification Queries

In [ ]:
%sql
SELECT COUNT(*) as total_isf_awards
FROM openalex.awards.isf_awards;


In [ ]:
%sql
-- Confirm the transformed table has the OpenAlex awards columns only.
DESCRIBE openalex.awards.isf_awards;


In [ ]:
%sql
-- Sample transformed awards.
SELECT
    id,
    display_name,
    funder_award_id,
    funder_scheme,
    funding_type,
    amount,
    currency,
    start_date,
    end_date,
    lead_investigator.given_name AS pi_given,
    lead_investigator.family_name AS pi_family,
    lead_investigator.affiliation.name AS institution,
    landing_page_url
FROM openalex.awards.isf_awards
LIMIT 10;


In [ ]:
%sql
-- Check ID and native-key uniqueness.
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT id) AS distinct_openalex_ids,
    COUNT(DISTINCT funder_award_id) AS distinct_funder_award_ids
FROM openalex.awards.isf_awards;


In [ ]:
%sql
-- Check funder distribution. Should be only F4320322252.
SELECT funder.display_name, funder_id, COUNT(*) as cnt
FROM openalex.awards.isf_awards
GROUP BY funder.display_name, funder_id
ORDER BY cnt DESC;


In [ ]:
%sql
-- Data completeness.
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(description) as has_description,
    COUNT(amount) as has_amount,
    COUNT(start_date) as has_start_date,
    COUNT(end_date) as has_end_date,
    COUNT(landing_page_url) as has_url,
    COUNT(lead_investigator.affiliation.name) as has_institution,
    COUNT(lead_investigator.family_name) as has_pi_family,
    SUM(amount) as total_funding,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) as pct_with_amount,
    ROUND(try_divide(COUNT(description), COUNT(*)) * 100.0, 1) as pct_description,
    ROUND(try_divide(COUNT(lead_investigator.family_name), COUNT(*)) * 100.0, 1) as pct_pi_family
FROM openalex.awards.isf_awards;


In [ ]:
%sql
-- Amount/currency check. Amount is the official approved annual budget; zero source values remain NULL.
SELECT
    COUNT(*) AS total,
    SUM(CASE WHEN amount > 0 THEN 1 ELSE 0 END) AS rows_with_positive_amount,
    ROUND(try_divide(SUM(CASE WHEN amount > 0 THEN 1 ELSE 0 END), COUNT(*)) * 100.0, 1) AS pct_positive_amount,
    COUNT(DISTINCT currency) AS distinct_currencies,
    collect_set(currency) AS currencies,
    ROUND(MIN(amount), 0) AS min_amount,
    ROUND(MAX(amount), 0) AS max_amount,
    ROUND(AVG(CASE WHEN amount > 0 THEN amount END), 0) AS avg_positive_amount,
    ROUND(SUM(amount), 0) AS total_amount
FROM openalex.awards.isf_awards;


In [ ]:
%sql
-- Year distribution.
SELECT start_year, COUNT(*) as cnt, ROUND(SUM(amount), 0) as total_ils
FROM openalex.awards.isf_awards
WHERE start_year IS NOT NULL
GROUP BY start_year
ORDER BY start_year DESC;


In [ ]:
%sql
-- Funder scheme distribution.
SELECT funder_scheme, funding_type, COUNT(*) as cnt, ROUND(SUM(amount), 0) as total_ils
FROM openalex.awards.isf_awards
GROUP BY funder_scheme, funding_type
ORDER BY cnt DESC;


In [ ]:
%sql
-- Verify rows inserted into the raw awards table at priority 146.
SELECT
    priority,
    provenance,
    COUNT(*) as cnt,
    COUNT(DISTINCT funder_award_id) as distinct_funder_award_ids,
    ROUND(SUM(amount), 0) as total_ils
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'isf_grant_search' AND priority = 146
GROUP BY priority, provenance;


## Admin Handoff Notes

Contractor has no S3/Databricks access. Admin must upload the parquet with `scripts/local/isf_to_s3.py`, run this notebook on Databricks, inspect the Step 6 verification cells, then run the combined CreateAwards notebook. Do not add job YAML until the Databricks ingest is run and QA is complete.
